# GMM-CT: Reconstructing Moving Objects from X-ray Projections

**A Gaussian Mixture Model Approach to Dynamic Computed Tomography**

---

Daniel Burrows · Alan Turing Institute Interview · February 2026

*Algorithm: Multi-stage gradient optimisation of a physics-driven GMM inverse problem*

## The Problem: CT When Objects Move

**Standard CT** assumes a **static object** — reconstruction is then a well-posed linear inverse problem (filtered back-projection, SART, …).

**Dynamic CT** breaks this assumption:
- Objects move *and rotate* during scanning
- Projection data is inconsistent across views
- Standard reconstruction methods produce severe motion artefacts

**Our setting:** multiple objects undergoing *ballistic motion* (known acceleration) with *in-plane rotation*. We observe a handful of X-ray projections and must recover:

| Unknown | Symbol | Meaning |
|---|---|---|
| Attenuation & shape | $\alpha_k$, $U_k$ | How much each object absorbs X-rays, and its geometry |
| Initial velocity | $v_{0,k}$ | Where each object is heading |
| Angular velocity | $\omega_k$ | How fast each object spins |

## The Forward Model: Physics of X-ray Projection

Each object is a **Gaussian component** with time-varying centre $\mu_k(t)$ and covariance $\Sigma_k(t)$.

**Trajectory model** (ballistic + rotation):
$$
\mu_k(t) = x_{0,k} + v_{0,k}\,t + \tfrac{1}{2}\,a_{0,k}\,t^2
$$
$$
\Sigma_k(t)^{-1/2} = U_k\,R(\omega_k t)^\top
$$

**X-ray projection** along direction $\hat{r}$ is the line integral of the attenuation field:

$$
p(s, r, t) = \sum_{k=1}^{N} \frac{\sqrt{\pi}\,\alpha_k}{\|U_k R(\omega_k t)^\top \hat{r}\|} \exp\!\left(-\frac{\bigl(U_k R^\top(r-\mu_k)\bigr)^2_{\perp}}{\|U_k R^\top \hat{r}\|^2}\right)
$$

This has a **closed-form analytic expression** — no numerical quadrature needed.

In [ ]:
# The forward model: project a rotating GMM onto receiver arrays
# (from gmm_ct/core/forward_model.py)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({'font.size': 13, 'figure.dpi': 120})

from gmm_ct import GMM_reco, generate_true_param, construct_receivers, set_random_seeds
from gmm_ct.config.defaults import GRAVITATIONAL_ACCELERATION

device = torch.device('cpu')
set_random_seeds(40)

# --- Geometry: 1 source, 128 receivers spanning [-3, 3] ---
sources = [torch.tensor([-1.0, -1.0], dtype=torch.float64)]
receivers = construct_receivers(device, (128, 4.0, -3.0, 3.0))

print(f"Sources : {len(sources)} position(s)")
print(f"Receivers: {len(receivers[0])} detectors, x in [{receivers[0][0][0]:.1f}, {receivers[0][-1][0]:.1f}]")

## Generating Synthetic Data

We simulate ground-truth parameters and create projection data to reconstruct.

Think of this as: *"take a CT scan of N rotating, falling objects — can we recover the objects?"*

In [ ]:
# Generate ground-truth parameters for N=3 Gaussians
d, N = 2, 3

theta_true = generate_true_param(
    d, N,
    initial_location=torch.tensor([-8.0, 0.0], dtype=torch.float64, device=device),
    initial_velocity=torch.tensor([3.0, 2.0], dtype=torch.float64, device=device),
    initial_acceleration=torch.tensor([0.0, -GRAVITATIONAL_ACCELERATION], dtype=torch.float64, device=device),
    min_rot=-24.0, max_rot=-20.0,
    device=device
)

# Build model and generate projection data
model = GMM_reco(
    d=d, N=N,
    sources=sources, receivers=receivers,
    x0s=theta_true['x0s'], a0s=theta_true['a0s'],
    omega_min=-24.0, omega_max=-20.0,
    device=device,
)

t = torch.linspace(0, 0.5, 50, dtype=torch.float64, device=device)
proj_data = model.generate_projections(t, theta_true)

print(f"N = {N} Gaussians | {len(t)} time steps")
print(f"True omegas (rad/s): {[f'{w.item():.2f}' for w in theta_true['omegas']]}")
print(f"True v0s:\n  " + '\n  '.join([str(v.numpy().round(3)) for v in theta_true['v0s']]))

In [ ]:
# Visualise the sinogram (projection data over time)
sino = proj_data[0].detach().cpu().numpy()  # shape: (n_time, n_receivers)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sinogram
im = axes[0].imshow(sino.T, aspect='auto', origin='lower',
                     extent=[t[0].item(), t[-1].item(), -3, 3],
                     cmap='inferno')
plt.colorbar(im, ax=axes[0], label='Attenuation')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Receiver position')
axes[0].set_title('Sinogram: observed projections over time')

# Single time slice — what each detector reads
axes[1].plot(np.linspace(-3, 3, 128), sino[0], label='t=0.00 s', color='steelblue')
axes[1].plot(np.linspace(-3, 3, 128), sino[25], label='t=0.25 s', color='darkorange')
axes[1].plot(np.linspace(-3, 3, 128), sino[-1], label='t=0.50 s', color='crimson')
axes[1].set_xlabel('Receiver position')
axes[1].set_ylabel('Projected attenuation')
axes[1].set_title('Single projection slices')
axes[1].legend()

plt.tight_layout()
plt.show()

## The Algorithm: 4-Stage Optimisation Pipeline

The inverse problem is **highly non-convex** — naive gradient descent fails. We exploit the physics structure:

| Stage | Parameters | Method | Loss |
|---|---|---|---|
| **1. Trajectory** | $v_{0,k}$ | Multi-start L-BFGS on peak heights | L2 + Hungarian assignment |
| **1.5. Velocity refinement** | $v_{0,k}$ | Newton-Raphson / L-BFGS on analytic derivative | Closed-form |
| **2. Joint morphology** | $\alpha_k, U_k, \omega_k$ | Multi-start L-BFGS on full projections | Smooth L1 (Huber, $\beta{=}0.3$) |
| **3. $\omega$ grid search** | $\omega_k$ | Brute-force $\pm 3$ Hz at 0.1 Hz resolution | Smooth L1 |
| **4. Final refinement** | All morphology | L-BFGS polish | Smooth L1 |

**Key insight:** $v_0$ and $\omega$ are near-orthogonal in information content — trajectory controls *where* the peaks are, morphology controls *their shape*. This lets us decouple the optimisation.

## Stage 1: Trajectory Estimation via Peak Tracking

**Observation:** each Gaussian produces a local peak in the projection at receiver position $r^*(t)$.

Under ballistic motion, the peak trajectory is:
$$
r^*(t) \approx x_{0,k} + v_{0,k}\,t + \tfrac{1}{2}\,a_{0,k}\,t^2 \quad (\text{first-order approximation})
$$

We detect peaks empirically and optimise $v_0$ to match them — a much simpler sub-problem than fitting full projections.

The **Hungarian algorithm** handles the assignment problem: which peak belongs to which Gaussian?

In [ ]:
# Show what peak detection looks like on the sinogram
from scipy.signal import find_peaks

fig, ax = plt.subplots(figsize=(10, 4))
receiver_pos = np.linspace(-3, 3, 128)

im = ax.imshow(sino.T, aspect='auto', origin='lower',
               extent=[t[0].item(), t[-1].item(), -3, 3],
               cmap='inferno', alpha=0.85)

# Detect and overlay peaks for each time step
for i in range(0, len(t), 3):
    peaks, _ = find_peaks(sino[i], height=0.01, distance=5)
    if len(peaks) > 0:
        ax.scatter([t[i].item()] * len(peaks), receiver_pos[peaks],
                   c='cyan', s=12, zorder=5, linewidths=0)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Receiver position')
ax.set_title('Sinogram with detected peaks (cyan) — Stage 1 input')
plt.colorbar(im, ax=ax, label='Attenuation')
plt.tight_layout()
plt.show()
print("Stage 1 optimises v₀ to match these parabolic peak tracks.")

## Live Reconstruction: N = 3

Running the full 4-stage pipeline on the data we just generated.

*This takes ~30–60 seconds on CPU for 3 Gaussians.*

In [ ]:
from time import time

print("Starting reconstruction...")
t0 = time()
theta_est = model.fit(proj_data, t)
elapsed = time() - t0
print(f"\nReconstruction complete in {elapsed:.1f}s")
print(f"Estimated omegas (rad/s): {[f'{w.item():.3f}' for w in theta_est['omegas']]}")
print(f"True     omegas (rad/s): {[f'{w.item():.3f}' for w in theta_true['omegas']]}")

In [ ]:
# Compare true vs estimated projections
proj_est = model.generate_projections(t, theta_est)

sino_est = proj_est[0].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

vmin, vmax = sino.min(), sino.max()
kw = dict(aspect='auto', origin='lower', cmap='inferno',
          extent=[t[0].item(), t[-1].item(), -3, 3], vmin=vmin, vmax=vmax)

im0 = axes[0].imshow(sino.T, **kw)
axes[0].set_title('Observed (ground truth)')

im1 = axes[1].imshow(sino_est.T, **kw)
axes[1].set_title('Reconstructed')

residual = np.abs(sino - sino_est)
im2 = axes[2].imshow(residual.T, aspect='auto', origin='lower',
                      extent=[t[0].item(), t[-1].item(), -3, 3], cmap='hot')
axes[2].set_title(f'|Residual|  (max={residual.max():.4f})')

for ax in axes:
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Receiver position')

plt.colorbar(im0, ax=axes[0])
plt.colorbar(im1, ax=axes[1])
plt.colorbar(im2, ax=axes[2])

plt.suptitle('Sinogram comparison: observed vs reconstructed (N=3)', fontsize=14)
plt.tight_layout()
plt.show()

## Parameter Recovery: How Accurate Are We?

We match estimated Gaussians to true ones via the Hungarian algorithm (on parameter distance), then compute errors.

In [ ]:
from gmm_ct.visualization.publication import reorder_theta_to_match_true

theta_est_matched = reorder_theta_to_match_true(theta_true, theta_est, N)

print(f"{'Parameter':<20} {'True':>20} {'Estimated':>20} {'Error':>12}")
print("-" * 76)

for k in range(N):
    v_true = theta_true['v0s'][k].numpy()
    v_est  = theta_est_matched['v0s'][k].detach().numpy()
    v_err  = np.linalg.norm(v_true - v_est)
    print(f"v₀[{k}]  (m/s)      {str(v_true.round(3)):>20} {str(v_est.round(3)):>20} {v_err:>10.4f}")

print()
for k in range(N):
    w_true = theta_true['omegas'][k].item()
    w_est  = theta_est_matched['omegas'][k].item()
    print(f"ω[{k}]   (rad/s)    {w_true:>20.4f} {w_est:>20.4f} {abs(w_true-w_est):>10.4f}")

print()
for k in range(N):
    a_true = theta_true['alphas'][k].item()
    a_est  = theta_est_matched['alphas'][k].item()
    print(f"α[{k}]   (atten.)   {a_true:>20.4f} {a_est:>20.4f} {abs(a_true-a_est):>10.4f}")

## Pre-computed Results: N = 5 Objects

Running N=5 takes longer — loading pre-computed results from saved experiments.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

# Load N=5 pre-computed reconstruction results
results_dir = Path('data/results/20260222_220307_N5')
results = torch.load(results_dir / 'results.pt', map_location='cpu', weights_only=False)

theta_true_N5 = results['theta_true']
theta_est_N5  = results['theta_est']

print(f"N=5 experiment loaded.")
print(f"True  omegas: {[f'{w.item():.3f}' for w in theta_true_N5['omegas']]}")
print(f"Est.  omegas: {[f'{w.item():.3f}' for w in theta_est_N5['omegas']]}")

In [ ]:
# Show the publication-quality reconstruction figure saved during the experiment
display(Image(filename=str(results_dir / 'individual_gaussian_reconstruction.pdf'), width=900))

In [ ]:
# Recovered parameter error bars across the N=5 Gaussians
from gmm_ct.visualization.publication import reorder_theta_to_match_true

N5 = 5
theta_est_N5_matched = reorder_theta_to_match_true(theta_true_N5, theta_est_N5, N5)

params = ['v0x', 'v0y', 'omega', 'alpha']
errors = {p: [] for p in params}

for k in range(N5):
    v_t = theta_true_N5['v0s'][k].numpy()
    v_e = theta_est_N5_matched['v0s'][k].detach().numpy()
    errors['v0x'].append(abs(v_t[0] - v_e[0]))
    errors['v0y'].append(abs(v_t[1] - v_e[1]))
    errors['omega'].append(abs(theta_true_N5['omegas'][k].item() - theta_est_N5_matched['omegas'][k].item()))
    errors['alpha'].append(abs(theta_true_N5['alphas'][k].item() - theta_est_N5_matched['alphas'][k].item()))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
labels = [f'G{k}' for k in range(N5)]
colors = ['steelblue', 'darkorange', 'crimson', 'mediumseagreen']

for ax, (param, label, color) in zip(axes, [('v0x', 'Δv₀ₓ (m/s)', 'steelblue'),
                                              ('v0y', 'Δv₀ᵧ (m/s)', 'darkorange'),
                                              ('omega', 'Δω (rad/s)', 'crimson'),
                                              ('alpha', 'Δα', 'mediumseagreen')]):
    bars = ax.bar(labels, errors[param], color=color, alpha=0.8, edgecolor='white')
    ax.set_ylabel(label)
    ax.set_title(f'|Error| {label}')
    ax.set_ylim(bottom=0)
    # Annotate bars
    for bar, val in zip(bars, errors[param]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Parameter recovery errors for N=5 Gaussians', fontsize=14)
plt.tight_layout()
plt.show()

## Why Is This Interesting?

**What makes the algorithm novel:**
- Combines a **physics-driven forward model** (analytic GMM projections) with **data-driven optimisation**
- The **decoupled multi-stage pipeline** sidesteps the non-convexity that defeats end-to-end approaches
- **Hungarian assignment** automatically solves the label-switching problem
- Works from **very limited data** (1 source, 1 receiver array)

**Connections to ATI research themes:**
- Inverse problems and uncertainty quantification
- Physics-informed machine learning
- Signal processing and imaging science

## Limitations and Future Directions

**Current limitations:**
- Assumes known $x_0$ and $a_0$ (positions and accelerations given)
- Slow for large $N$ — runtime scales as $O(N \times \text{n\_sources} \times \text{n\_rcvrs} \times T)$
- Relies on multi-start: expensive, but necessary for non-convex landscape
- 2D only — no through-plane rotation

**Future directions:**
1. **3D extension** — full $SO(3)$ rotation parameterisation via Cayley maps
2. **Learned initialisations** — amortise multi-start cost with a neural network warm-start
3. **Uncertainty quantification** — Laplace approximation around the final optimum
4. **Real experimental data** — high-speed synchrotron CT of fragmenting projectiles

---

**Thank you.** Questions welcome.